In [1]:
# 01_data.ipynb — Data Ingestion and Quality Control

# Load a small public single-cell dataset (PBMC 3k), perform basic quality-control,
# normalization, and create reproducible train/validation/test splits.
# Artifacts are saved under `artifacts/`.


In [2]:
import os, json, numpy as np, pandas as pd, scanpy as sc

# create artifacts directory
ARTIFACTS_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

sc.settings.verbosity = 2
print("Artifacts directory:", ARTIFACTS_DIR)


Artifacts directory: /Users/anuragdani/Github/neuroai-project/notebooks/artifacts


In [3]:
# --- Cell 3: Load PBMC dataset with robust fallbacks (no hard-coded URLs) ---
import os
import scanpy as sc
import anndata as ad
import numpy as np

# local cache (works even if upstream mirrors change later)
cache_dir = os.path.join("data", "scanpy_cache")
os.makedirs(cache_dir, exist_ok=True)
sc.settings.cachedir = cache_dir
sc.settings.verbosity = 2

adata = None
errors = []

# 1) Primary: PBMC 3k (Scanpy-maintained mirror)
try:
    adata = sc.datasets.pbmc3k()
    print(f"Loaded PBMC3k via scanpy.datasets.pbmc3k(): {adata.n_obs} cells × {adata.n_vars} genes")
except Exception as e:
    errors.append(("pbmc3k", str(e)))

# 2) Secondary: PBMC 3k (processed variant)
if adata is None:
    try:
        adata = sc.datasets.pbmc3k_processed()
        print(f"Loaded PBMC3k (processed) via scanpy.datasets.pbmc3k_processed(): "
              f"{adata.n_obs} cells × {adata.n_vars} genes")
    except Exception as e:
        errors.append(("pbmc3k_processed", str(e)))

# 3) Tertiary: PBMC 68k reduced (tiny, good for demos)
if adata is None:
    try:
        adata = sc.datasets.pbmc68k_reduced()
        print(f"Loaded PBMC68k (reduced) via scanpy.datasets.pbmc68k_reduced(): "
              f"{adata.n_obs} cells × {adata.n_vars} genes")
    except Exception as e:
        errors.append(("pbmc68k_reduced", str(e)))

# 4) Final fallback: synthetic blobs (keeps pipeline working offline)
if adata is None:
    print("All remote loads failed; generating synthetic PBMC-like blobs so the pipeline can proceed.")
    # ~3000 cells, ~2000 genes, 5 clusters — similar shape to PBMC3k flow
    adata = sc.datasets.blobs(
        n_observations=3000,
        n_variables=2000,
        n_centers=5,
        cluster_std=2.5,
        random_state=42,
    )
    # blobs() creates X plus .obs['blobs']; ensure gene/cell names exist
    adata.var_names = [f"Gene{i}" for i in range(adata.n_vars)]
    adata.obs_names = [f"Cell{i}" for i in range(adata.n_obs)]
    print(f"Synthetic dataset: {adata.n_obs} cells × {adata.n_vars} features")

# Show any errors encountered (for debugging, optional)
if errors:
    print("Load attempts that failed (safe to ignore if we loaded something):")
    for name, err in errors:
        print(f" - {name}: {err}")

adata


try downloading from url
https://falexwolf.de/data/pbmc3k_raw.h5ad
... this may take a while but only happens once
Loaded PBMC3k (processed) via scanpy.datasets.pbmc3k_processed(): 2638 cells × 1838 genes
Load attempts that failed (safe to ignore if we loaded something):
 - pbmc3k: <urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)>


AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_cells'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [ ]:
# --- Cell 6: Create pseudo-labels and reproducible splits (safe version) ---

from sklearn.model_selection import train_test_split
import numpy as np, json, os, scanpy as sc

# only recompute embeddings if they aren't already present
if "neighbors" not in adata.uns:
    sc.pp.pca(adata)
    sc.pp.neighbors(adata)

# run Leiden if 'label' not already there
if "label" not in adata.obs.columns:
    sc.tl.leiden(adata, resolution=0.5, key_added="label")
else:
    print("Existing cluster labels detected; skipping recomputation.")

labels = adata.obs["label"].astype("category").cat.codes.values
idx = np.arange(adata.n_obs)

# check class counts before stratifying
unique, counts = np.unique(labels, return_counts=True)
if np.min(counts) < 2 or len(unique) < 2:
    print("Too few samples per class for stratified split; using random split.")
    stratify = None
else:
    stratify = labels

# 64/16/20 train/val/test
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=42, stratify=stratify
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.2, random_state=42,
    stratify=stratify[train_idx] if stratify is not None else None
)

splits = {
    "train_idx": train_idx.tolist(),
    "val_idx": val_idx.tolist(),
    "test_idx": test_idx.tolist(),
    "seed": 42,
}
os.makedirs("artifacts", exist_ok=True)
with open(os.path.join("artifacts", "splits.json"), "w") as f:
    json.dump(splits, f)

print("Split sizes:", {k: len(v) for k, v in splits.items() if "idx" in k})


running Leiden clustering


/var/folders/xs/mvlhf_hs219czt07s1jwvby00000gn/T/ipykernel_85690/1479541817.py:13: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=0.5, key_added="label")


    finished (0:00:00)
Split sizes: {'train_idx': 1688, 'val_idx': 422, 'test_idx': 528}


In [5]:
# --- Cell 4: Basic quality metrics summary ---

import pandas as pd, numpy as np, os, json

qc = {
    "n_obs": int(adata.n_obs),
    "n_vars": int(adata.n_vars),
    "sparsity": float((adata.X == 0).sum() / (adata.n_obs * adata.n_vars)),
}
pd.DataFrame([qc]).to_csv(os.path.join("artifacts", "qc_summary.csv"), index=False)
print("QC summary:", qc)


QC summary: {'n_obs': 2638, 'n_vars': 1838, 'sparsity': 0.0}


In [6]:
# --- Cell 5: Normalize counts and select highly variable genes (robust version) ---

import scanpy as sc
import numpy as np

# detect whether data is already log-transformed
X = adata.X.A if hasattr(adata.X, "A") else adata.X
is_logged = np.median(X[X > 0]) < 20  # crude heuristic

if is_logged:
    print("Detected preprocessed/log-transformed data — skipping re-normalization.")
else:
    print("Normalizing raw counts per cell …")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

# Run HVG selection safely
try:
    sc.pp.highly_variable_genes(
        adata, n_top_genes=min(2000, adata.n_vars), subset=True, flavor="seurat"
    )
    print(f"Selected {adata.n_vars} highly variable genes.")
except ValueError as e:
    print("HVG selection warning:", e)
    print("Falling back to simple variance-based selection.")
    import pandas as pd

    var = np.var(X, axis=0)
    top_idx = np.argsort(var)[::-1][:2000]
    adata = adata[:, top_idx]
    adata.var["highly_variable"] = False
    adata.var.iloc[top_idx, adata.var.columns.get_loc("highly_variable")] = True

print(f"Post-HVG matrix shape: {adata.n_obs} cells × {adata.n_vars} genes")
adata


Detected preprocessed/log-transformed data — skipping re-normalization.
extracting highly variable genes
    finished (0:00:00)
Selected 1838 highly variable genes.
Post-HVG matrix shape: 2638 cells × 1838 genes


AnnData object with n_obs × n_vars = 2638 × 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'label'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups', 'label', 'hvg'
    obsm: 'X_pca', 'X_tsne', 'X_umap', 'X_draw_graph_fr'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [ ]:
# --- Cell 6: Create pseudo-labels and reproducible splits (safe version) ---

from sklearn.model_selection import train_test_split
import numpy as np, json, os, scanpy as sc

# only recompute embeddings if they aren't already present
if "neighbors" not in adata.uns:
    sc.pp.pca(adata)
    sc.pp.neighbors(adata)

# run Leiden if 'label' not already there
if "label" not in adata.obs.columns:
    sc.tl.leiden(adata, resolution=0.5, key_added="label")
else:
    print("Existing cluster labels detected; skipping recomputation.")

labels = adata.obs["label"].astype("category").cat.codes.values
idx = np.arange(adata.n_obs)

# check class counts before stratifying
unique, counts = np.unique(labels, return_counts=True)
if np.min(counts) < 2 or len(unique) < 2:
    print("Too few samples per class for stratified split; using random split.")
    stratify = None
else:
    stratify = labels

# 64/16/20 train/val/test
train_idx, test_idx = train_test_split(
    idx, test_size=0.2, random_state=42, stratify=stratify
)
train_idx, val_idx = train_test_split(
    train_idx, test_size=0.2, random_state=42,
    stratify=stratify[train_idx] if stratify is not None else None
)

splits = {
    "train_idx": train_idx.tolist(),
    "val_idx": val_idx.tolist(),
    "test_idx": test_idx.tolist(),
    "seed": 42,
}
os.makedirs("artifacts", exist_ok=True)
with open(os.path.join("artifacts", "splits.json"), "w") as f:
    json.dump(splits, f)

print("Split sizes:", {k: len(v) for k, v in splits.items() if "idx" in k})


Existing cluster labels detected; skipping recomputation.
Split sizes: {'train_idx': 1688, 'val_idx': 422, 'test_idx': 528}


In [8]:
# --- Cell 7: Save preprocessed AnnData object ---

out_path = os.path.join("artifacts", "preprocessed_small.h5ad")
adata.write_h5ad(out_path)
print(f"Saved processed dataset → {out_path}")


Saved processed dataset → artifacts/preprocessed_small.h5ad
